# Anthropic Apps — Claude Code & Computer Use

**Live online course — instructor walkthrough notebook**

This notebook follows the lecture notes about Anthropic's deployed applications. Each section has:
- **Lecture notes** (markdown) — what to explain on the slide/screen.
- **Demo code** — cells that reproduce, via the SDK, the same primitives that power the real apps.
- **🏫 During class** callouts — specific instructor actions, talking points, and live variations.

The framing for this whole notebook: **Claude Code and Computer Use are two reference implementations of the same building blocks we used in the Intro notebook** — `messages`, `tools`, `system` prompts, and multi-turn loops. By the end of this session, students should be able to point at any feature of Claude Code or Computer Use and identify the SDK primitive underneath.

---

## Agenda

1. Anthropic Apps — what they are
2. Claude Code Setup
3. Claude Code in Action
4. Enhancements with MCP Servers
5. Parallelizing Claude Code
6. Automated Debugging
7. Computer Use
8. How Computer Use Works
9. Recap + practice exercises


## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so it is never committed to version control.

> **🏫 During class:** Show the `.env` file (blur the key on screen). Restate the rule: **never paste the API key directly into a notebook cell** — `.env` + `python-dotenv` is the entire pattern for keeping secrets out of source control.


In [1]:
# Install packages (uncomment if not already installed)
%pip install anthropic python-dotenv



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


This setup cell does four things:

1. **`load_dotenv()`** — reads the `.env` file next to the notebook and copies its `KEY="value"` lines into process environment variables. After it runs, `os.getenv("ANTHROPIC_API_KEY")` returns your key without it ever appearing in notebook source.
2. **`client = anthropic.Anthropic()`** — creates the SDK client. No arguments on purpose: the SDK auto-discovers `ANTHROPIC_API_KEY` from the environment.
3. **`model = "claude-sonnet-4-6"`** — pin the default workhorse model in one constant. Every later demo references `model` instead of hard-coding the string, so you can swap one line and re-run the whole notebook.
4. **The three `print(...)` lines** — sanity check before any API call. If `Key loaded:` prints `False`, the `.env` is missing or misnamed — fix it before running anything below.


In [ ]:
from dotenv import load_dotenv
import anthropic
import os

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically

# Default workhorse model for the notebook.
model = "claude-haiku-20240307"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


SDK version: 0.96.0
Model: claude-sonnet-4-6
Key loaded: True


---
# 1. Anthropic Apps — what they are

**Anthropic Apps** = real applications shipped by Anthropic that run on top of the same SDK we have been using. Two flagship examples:

| App | Surface | Underlying SDK primitive |
|---|---|---|
| **Claude Code** | Terminal-based coding assistant | `messages`, `system` prompts, `tools` (file read/write, terminal, MCP) |
| **Computer Use** | Visual control of a desktop or browser | `tools` (a special `computer_*` tool type) + a runtime that executes actions |

### Why study them
1. They prove the SDK is enough to build production agents — every "magic" feature has a plain SDK equivalent.
2. They are reference implementations: the patterns Anthropic uses are documented and copyable.
3. They expose the *gap* between a single API call and an autonomous agent — the loop, the tools, the memory, the parallelism.

The rest of this notebook tours that gap, one primitive at a time.


### Demo: a single SDK call vs an agent

Before we build anything, ground the framing: **a single `messages.create()` call is one turn of conversation**. Claude Code wraps that call in a loop, attaches file-system tools, and persists context to `claude.md`. Computer Use wraps it in a loop with screenshot+click tools running inside Docker.

The cell below is just one call — exactly what the rest of the notebook will build on. Watch how short it is, and read Claude's answer with the rest of the agenda in mind.


In [3]:
response = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[{"role": "user", "content": (
        "In one paragraph: what's the difference between calling the Claude API once "
        "and running an autonomous agent like Claude Code?"
    )}],
)

print(response.content[0].text)


When you call the Claude API once, you send a prompt and get a single response back — it's a stateless, discrete transaction where you're in control of what happens next. An autonomous agent like Claude Code, by contrast, runs in a loop: it takes an action, observes the result, decides what to do next, and repeats this cycle many times, often across dozens or hundreds of API calls, using tools like file editing, terminal commands, and web search to make real changes in the world. The key differences are **persistence** (the agent maintains context and goals across many steps), **agency** (it decides its own next actions rather than waiting for you), and **consequence** (mistakes compound and can have real, hard-to-reverse effects on your codebase, system, or data — rather than just producing a slightly wrong piece of text you can ignore).


> **🏫 During class:**
> 1. Run the cell. Read Claude's answer aloud.
> 2. Say: *"Everything Claude just described — the loop, the tools, the memory — is what we're going to add to this single call, primitive by primitive."*
> 3. Question for the room: *"If the only difference between an API call and an agent is a `while` loop and some tools, why does an agent feel so much more powerful?"* (Surface answers; they'll come up again in §3 and §8.)


---
# 2. Claude Code Setup

**Claude Code** = a terminal-based coding assistant built by Anthropic. It can search/read/edit files, fetch web content, run terminal commands, and consume external tools via MCP.

### Setup recipe (from the docs)
1. Install Node.js (`npm help` confirms it works).
2. `npm install -g @anthropic-ai/claude-code`.
3. Run `claude` in a terminal to authenticate against your Anthropic account.

Full docs: **docs.anthropic.com**.

### What's actually new vs. the SDK
Claude Code is *not* a different model — it's the same Claude reachable via `client.messages.create()`. What you get on top:

| Capability | What's happening underneath |
|---|---|
| Read/edit files | A file-system tool exposed via the `tools=` parameter |
| Run terminal commands | A shell tool exposed via `tools=` |
| Web fetch | A browser/HTTP tool exposed via `tools=` |
| MCP client | Same `tools=` slot, populated dynamically from MCP servers |

The "assistant persona" is a system prompt. The "loop until done" is a `while` loop you could write in 30 lines.


### Demo: a Claude Code-shaped system prompt

The simplest piece of Claude Code we can reproduce in a notebook: take a project's README, hand it to Claude under a "Claude Code"-flavored system prompt, and watch it produce setup commands the way Claude Code does on first launch.

Watch how a single system prompt reframes the model from a generic chatbot into a setup assistant. The user message hasn't changed — only the role.


In [4]:
readme = """\
# Sample App

## Setup
- Requires Python 3.11+ and `uv`
- Copy `.env.example` to `.env` and fill in `ANTHROPIC_API_KEY`
- Install with `uv pip install -e .`
- Run with `uv run main.py`
"""

claude_code_system = (
    "You are Claude Code, a terminal-based coding assistant. "
    "When given a project's README, output the exact shell commands needed to set the project up "
    "from a fresh clone — one command per line, no commentary, no markdown fences."
)

response = client.messages.create(
    model=model,
    max_tokens=300,
    system=claude_code_system,
    messages=[{"role": "user", "content": readme}],
)

print(response.content[0].text)


cp .env.example .env
uv pip install -e .
uv run main.py


> **🏫 During class:**
> 1. Run the cell. Read out the commands Claude generated.
> 2. Say: *"What you just saw is essentially Claude Code's startup move — read the README, suggest setup. The whole tool is a souped-up version of this single call."*
> 3. Variation to try live: edit the README to drop the `uv` requirement, re-run. Point out how the generated commands shift instantly — the system prompt fixes the *style*, not the *content*.


---
# 3. Claude Code in Action

In real use, Claude Code behaves like a **collaborative engineer**, not a code generator. The recommended workflow is **three turns**, not one:

1. **Identify** — point Claude at the relevant files; ask which matter for the task. **No code yet.**
2. **Plan** — describe the feature; ask Claude to plan a solution. **Still no code.**
3. **Implement** — ask Claude to write the patch.

### Why three turns and not one
Asking Claude to "add 2FA to login" in a single shot makes it skip both file selection and design. The three-turn split forces a moment of reflection between each stage and lets you correct course before any code is written.

### Other Claude Code features worth naming
- **`init`** — Claude scans the codebase and writes a `claude.md` with architecture notes; that file is auto-included as context for future requests.
- **Memory tiers** — Project (shared, committed), Local (user-only, in same repo), User (across all repos).
- **`#` syntax** — type `# note text` in chat to append to memory; same effect as editing `claude.md` by hand.
- **Test-driven workflow** — alternative to the three-step: ask for tests first, then ask Claude to make them pass.

To demo the three-turn workflow we need helpers that maintain message history (the SDK is stateless, so we maintain state ourselves).


### Helpers we'll reuse for the rest of the notebook

We'll define `add_user_message`, `add_assistant_message`, and `chat`. From here onward, every text-only demo uses `chat(...)` so cells stay small and the *one varying argument* is obvious.

`chat` also disables Sonnet 4.6's adaptive extended thinking. Thinking is great for hard reasoning, but it's mutually exclusive with **assistant pre-fill** (which we'll need in §6 to force JSON output). Disabling it once here keeps every later demo consistent.


In [5]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        # Sonnet 4.6 / Opus 4.7 default to adaptive extended thinking, which is
        # mutually exclusive with assistant-message pre-fill (used in §6).
        # Disable it here so every demo in the notebook behaves consistently.
        "thinking": {"type": "disabled"},
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


### Demo: three-turn workflow on a fictional repo

We hand Claude a list of files plus a task ("add 2FA to login") and walk three turns:
1. **Identify** the relevant files.
2. **Plan** the change.
3. **Implement** a minimal patch.

Watch how each turn references the previous turn's answer — that only works because we re-send the full message history. Skip turn 1 or 2 and the implementation drifts toward generic boilerplate.


In [6]:
files = """\
- src/auth.py — login & signup handlers
- src/cart.py — shopping cart logic
- src/payment.py — Stripe integration
- src/notifications.py — email/SMS sender
- tests/test_auth.py
- README.md
"""

msgs = []

# Turn 1 — identify
add_user_message(msgs, (
    f"Project files:\n{files}\n"
    "Task: 'Add two-factor authentication to login.' "
    "Which files matter? Don't write code yet — just list the files and a one-line reason for each."
))
identify = chat(msgs)
add_assistant_message(msgs, identify)
print("--- Step 1 (identify) ---")
print(identify, "\n")

# Turn 2 — plan
add_user_message(msgs, "Now propose a 5-bullet implementation plan. Still no code.")
plan = chat(msgs)
add_assistant_message(msgs, plan)
print("--- Step 2 (plan) ---")
print(plan, "\n")

# Turn 3 — implement
add_user_message(msgs, "Now write a minimal pseudo-code patch for the most important file in your list.")
patch = chat(msgs)
add_assistant_message(msgs, patch)
print("--- Step 3 (implement) ---")
print(patch)


--- Step 1 (identify) ---
## Relevant Files

| File | Reason |
|------|--------|
| `src/auth.py` | **Primary target** — login handler is where 2FA verification logic gets added |
| `src/notifications.py` | Needs to send the OTP code via email or SMS to the user |
| `tests/test_auth.py` | Must be updated to cover the new 2FA flow and edge cases |
| `README.md` | Should document the new authentication requirement for users/developers |

---

## Files You Can Ignore

| File | Reason |
|------|--------|
| `src/cart.py` | Completely unrelated to authentication |
| `src/payment.py` | Stripe integration has no dependency on login flow |

---

## One Thing Worth Flagging

You'll likely also need a **new file or a database migration** (e.g. `src/models.py` or a schema change) to store things like:
- The user's 2FA preference (enabled/disabled)
- The OTP secret or temporary code
- Code expiry timestamp

That dependency isn't visible in the current file list, but it's worth confirming before writ

> **🏫 During class:**
> 1. Run all three turns. Read each in order and point out how Step 3's code references files Claude already named in Step 1.
> 2. Say: *"We're the loop. The model is stateless. Every turn re-sends the entire history — that's why context carries."*
> 3. Variation to try live: re-run with `msgs = []` re-initialized between calls (so each turn forgets) and watch Step 3 collapse into generic 2FA boilerplate. That collapse is the whole point of three turns.


---
# 4. Enhancements with MCP Servers

**MCP** = Model Context Protocol. An MCP **server** publishes tools (functions); an MCP **client** (Claude Code, in this case) connects to it and registers those tools dynamically.

### Add a server to Claude Code
```bash
claude mcp add docs uv run main.py
```
Now Claude Code has every tool that `main.py` exposes — for example, `document_path_to_markdown`, which takes a `.pdf`/`.docx` and returns Markdown text.

### Common MCP servers in real workflows
- **Sentry** — production error monitoring
- **Jira / Linear** — ticket management
- **Slack** — team messaging
- **GitHub** — issues, PRs, commits
- **Custom internal tools** — anything your team has scripted

### What's the SDK equivalent
Exactly the same as **tool use**: you pass a list of tool schemas via `tools=...`, Claude returns a `tool_use` block when it wants to call one, and **you** execute the call and send the result back.

MCP is a *transport* on top of that protocol — instead of you hard-coding the schema in your script, MCP fetches it from a running server. The wire format Claude sees is identical.


### Demo: an MCP-style tool, but with a hard-coded schema

This cell defines the same tool the lecture notes describe — `document_path_to_markdown` — directly in the SDK call. No MCP server, no separate process. We send a user message that the tool can solve, and watch Claude's response.

The interesting bit is the **shape of the response**: not text, but a `tool_use` block. `stop_reason='tool_use'` is Claude saying "I want you to run this and call me back." That round-trip is the entire MCP protocol in disguise.


In [10]:
tools = [
    {
        "name": "document_path_to_markdown",
        "description": "Convert a local document file (PDF, DOCX, etc.) to Markdown text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "Absolute path to the document file on disk.",
                },
            },
            "required": ["path"],
        },
    }
]

response = client.messages.create(
    model=model,
    max_tokens=500,
    tools=tools,
    messages=[{"role": "user", "content": (
        "Please read /Users/me/quarterly_report.pdf and tell me what its top-level headings are."
    )}],
)

print("stop_reason:", response.stop_reason)
print()
for block in response.content:
    print("--- block type:", block.type, "---")
    print(block)


stop_reason: tool_use

--- block type: tool_use ---
ToolUseBlock(id='toolu_011takuXuVhjyn2MGUBGkVa6', caller=DirectCaller(type='direct'), input={'path': '/Users/me/quarterly_report.pdf'}, name='document_path_to_markdown', type='tool_use')


> **🏫 During class:**
> 1. Run the cell. Point out `stop_reason='tool_use'` and the `ToolUseBlock` with `name`, `id`, and `input={'path': '...'}`.
> 2. Say: *"This is the same protocol Claude Code uses with every MCP tool. The difference is just where the schema came from — we hand-wrote it; an MCP server would have published it over stdio."*
> 3. Question for the room: *"What would Claude Code do next? What would happen if we don't actually run the tool and just send back made-up text?"* (Tease §8 — that's where we close the loop.)


---
# 5. Parallelizing Claude Code

Running **multiple Claude instances simultaneously** scales productivity — one developer commands a virtual team. The blocker is simple: two instances editing the same file collide.

### The Claude Code answer: git worktrees
A **worktree** is a separate directory tied to a separate branch of the same repo. Each Claude instance works in its own worktree, on its own branch, in isolation. When done, branches are merged back; Claude resolves conflicts.

```
main repo
  ├── worktree-1  (branch: feature-a)  ← Claude instance 1
  ├── worktree-2  (branch: feature-b)  ← Claude instance 2
  └── worktree-3  (branch: feature-c)  ← Claude instance 3
```

Custom commands in `.claude/commands/<name>.md` (with a `$ARGUMENTS` placeholder) can automate the create-worktree-and-launch dance.

### What's the SDK equivalent
Different *target*, same *primitive*: each parallel call is an **independent message list**. There's no shared state on the API side, so two `messages.create()` calls running concurrently can't conflict. The SDK is already safe for parallel use.

The Claude Code worktree pattern is the same idea applied to *files* instead of *messages*: give each agent its own isolated workspace.


### Demo: sequential vs parallel SDK calls

We send three independent prompts and time two strategies:
1. **Sequential** — each call waits for the previous to finish.
2. **Parallel** — all three calls launched concurrently via `ThreadPoolExecutor`.

Expect the parallel version to be roughly N× faster (where N = number of tasks), because each call mostly waits on the network. This is the SDK-level analogue of running three Claude Code instances on three worktrees.


In [11]:
import time
from concurrent.futures import ThreadPoolExecutor

tasks = [
    "Write a one-line Python function that reverses a string.",
    "Write a one-line SQL query that counts rows in 'users'.",
    "Write a one-line shell command that prints today's date.",
]

# Sequential
start = time.time()
sequential_replies = [chat([{"role": "user", "content": t}]) for t in tasks]
sequential_time = time.time() - start

# Parallel
def run_one(task):
    return chat([{"role": "user", "content": task}])

start = time.time()
with ThreadPoolExecutor(max_workers=len(tasks)) as pool:
    parallel_replies = list(pool.map(run_one, tasks))
parallel_time = time.time() - start

print(f"Sequential: {sequential_time:.2f}s")
print(f"Parallel:   {parallel_time:.2f}s")
print(f"Speedup:    {sequential_time / parallel_time:.2f}x")
print()
for t, r in zip(tasks, parallel_replies):
    print(f"[{t[:35]}...]")
    print(f"  -> {r.strip()[:120]}")
    print()


Sequential: 5.56s
Parallel:   3.09s
Speedup:    1.80x

[Write a one-line Python function th...]
  -> Here's a one-line Python function that reverses a string using slicing:

```python
reverse_string = lambda s: s[::-1]
``

[Write a one-line SQL query that cou...]
  -> ```sql
SELECT COUNT(*) FROM users;
```

[Write a one-line shell command that...]
  -> ```bash
date
```

This prints the current date and time. If you want **just the date** (e.g., `2024-01-15`):

```bash
da



> **🏫 During class:**
> 1. Run the cell. Speedup should be near 3×.
> 2. Say: *"Three independent calls take roughly the same wall-clock time as one. That's the same productivity multiplier git worktrees give Claude Code — applied to API calls instead of repos."*
> 3. Variation: bump `tasks` to 10 items, raise `max_workers` to 10, re-run. Speedup grows linearly until you hit the API rate limit. (That's exactly the cap on real Claude Code parallelism, too — the developer's ability to *manage* simultaneous agents.)


---
# 6. Automated Debugging

**Idea:** wire Claude into a CI loop so production errors are detected, analyzed, and PR'd without a human touching a log file.

### The reference workflow
1. **GitHub Action** runs daily at 9am UTC.
2. AWS CLI fetches the last 24h of CloudWatch logs.
3. **Claude** parses logs, identifies unique errors, deduplicates.
4. Claude proposes a fix as a structured object (file + diff + explanation).
5. Action runs `gh pr create` with the proposed change.

### Why this matters
Production environments diverge from development in ways unit tests miss — bad model IDs, expired keys, wrong region settings. These show up only in real traffic, only in real logs. A daily AI-driven sweep catches them before users complain.

### What's the SDK equivalent
Step 3 — the analysis step — is one `messages.create()` call. The trick is **structured output**: we want a JSON object back, not free-form prose. We use the same pre-fill + stop-sequence pattern from the Intro notebook.


### Demo: simulate the analysis step

We feed Claude a fake CloudWatch log slice that contains a deprecated-model error (the canonical "works locally, breaks in prod" failure). Pre-fill the assistant turn with `` ```json ``, stop on `` ``` ``, and Claude returns directly parseable JSON.

Watch the JSON shape — `error`, `root_cause`, `fix` (file + suggested change). That object is exactly what the GitHub Action would feed into a PR description.


In [12]:
import json

logs = """\
2026-04-30T14:01:22Z ERROR app.api: anthropic.NotFoundError: model not found: claude-3-haiku-20240307
2026-04-30T14:03:11Z ERROR app.api: anthropic.NotFoundError: model not found: claude-3-haiku-20240307
2026-04-30T14:08:05Z ERROR app.api: anthropic.NotFoundError: model not found: claude-3-haiku-20240307
2026-04-30T14:09:12Z INFO  app.api: shutting down
"""

system = (
    "You are an automated debugging agent. Read production logs, identify the unique error "
    "(de-duplicate identical messages), and return ONE JSON object with keys: "
    "error, root_cause, fix. The 'fix' value must be an object with keys 'file' and 'suggested_change'."
)

msgs = [
    {"role": "user", "content": f"Logs from the last 24 hours:\n{logs}"},
    {"role": "assistant", "content": "```json"},
]

raw = chat(msgs, system=system, stop_sequences=["```"])
print("--- Raw model output ---")
print(raw)
print()

parsed = json.loads(raw)
print("--- Parsed object ---")
print(json.dumps(parsed, indent=2))


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011Cacj2qZ54AnaqiooeTgDX'}

> **🏫 During class:**
> 1. Run the cell. Show the raw output, then the parsed dict — point out it's already valid JSON, no string cleanup.
> 2. Say: *"This is the entire 'AI' part of an automated-debugging pipeline. Wrap it in a cron job and a `gh pr create`, and you have the workflow from the lecture notes."*
> 3. Variation: change the log to mix two distinct errors and re-run — Claude only reports one. Ask the room: *"How would we change the prompt to get a list of all unique errors?"* (Hint: change the schema to `{"errors": [...]}`.)


---
# 7. Computer Use

**Computer Use** = Claude can control a screen — take screenshots, move the mouse, click, type, and follow multi-step UI instructions autonomously.

### Capabilities
- Screenshot any application or browser.
- Click at specific coordinates.
- Type text into focused fields.
- Read what's on screen visually (image input).
- Chain those actions to complete a task.

### Reference setup
Anthropic ships a Docker container with the mouse/keyboard execution code pre-built. Pull it, run it, chat with Claude through a small UI, watch it drive a browser inside the container.

### Use cases
- Automated **QA testing** of web apps (the marquee one).
- Repetitive UI work where no API exists.
- Bug hunting via systematic clicking through unfamiliar interfaces.

### Production caveats
Visual understanding is imperfect: small icons, dense layouts, and rapidly changing pages still trip it up. Treat it like a junior tester who needs clear instructions and verifiable success criteria.


### Demo: a computer-use-shaped tool schema

The real Anthropic computer-use tool uses a special pre-defined tool type (`"computer_20250124"`) that auto-supplies the action vocabulary. To keep this notebook portable and runnable without Docker, we'll define an **equivalent custom tool** with the same shape — `action`, optional `x`/`y`/`text`. The protocol Claude uses is identical.

We'll send a single instruction (*"Take a screenshot of the current window"*) and observe the `tool_use` Claude returns.


In [ ]:
computer_tool = {
    "name": "computer_action",
    "description": (
        "Perform an action on the computer screen. "
        "Use 'screenshot' to capture, 'left_click' with x/y to click, 'type' with text to type."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "action": {
                "type": "string",
                "enum": ["screenshot", "left_click", "type"],
            },
            "x": {"type": "number", "description": "Pixel x-coordinate (for click)."},
            "y": {"type": "number", "description": "Pixel y-coordinate (for click)."},
            "text": {"type": "string", "description": "Text to type (for type action)."},
        },
        "required": ["action"],
    },
}

response = client.messages.create(
    model=model,
    max_tokens=500,
    tools=[computer_tool],
    messages=[{"role": "user", "content": "Take a screenshot of the current window."}],
)

print("stop_reason:", response.stop_reason)
print()
for block in response.content:
    print("--- block type:", block.type, "---")
    print(block)


> **🏫 During class:**
> 1. Run the cell. Highlight: `stop_reason='tool_use'`, the `ToolUseBlock` with `name='computer_action'` and `input={'action': 'screenshot'}`.
> 2. Say: *"Computer Use is not a special model — it's a tool schema plus a runtime that knows how to execute it. The schema is what we just wrote; the runtime is the Docker container Anthropic ships."*
> 3. Variation: change the prompt to *"Click the Sign In button at coordinates 420, 280"* and re-run. Watch the `input` dict change to `{'action': 'left_click', 'x': 420, 'y': 280}`. Same tool, different input — that's the whole API surface.


---
# 8. How Computer Use Works

Computer Use is **tool use with a different runtime**. The full round-trip:

```
                    1. user: "click the Sign In button"
                          |
                          v
        +------------------------------+
        |           Claude              |
        +------------------------------+
                          |
                    2. tool_use: {action: "screenshot"}
                          |
                          v
        +------------------------------+
        |   Your runtime (Docker)       |   <-- takes screenshot
        +------------------------------+
                          |
                    3. tool_result: <png bytes>
                          |
                          v
        +------------------------------+
        |           Claude              |   <-- sees the screen
        +------------------------------+
                          |
                    4. tool_use: {action: "left_click", x: 420, y: 280}
                          |  ... round-trip continues until done ...
```

### Key design points
- **Claude does not directly manipulate any computer.** It emits structured tool calls; *your* runtime executes them.
- The runtime is yours to build (or use Anthropic's reference Docker image).
- Each loop iteration is one `messages.create()` call. State persists because *you* re-send the message history each time.

### What we'll demo
We'll run two `messages.create()` calls in sequence:
- Call 1 → Claude emits a `tool_use` for an action.
- We append a fake `tool_result` (simulating what the runtime would observe).
- Call 2 → Claude continues, now grounded in our "observation."

That's the whole loop. A real Computer Use session is just this, repeated until Claude says it's done.


### Demo: a two-call round-trip with a simulated tool result

Call 1 sends a navigation-style task. Claude requests an action. We pretend the action ran and feed back a plausible observation. Call 2 lets Claude continue with that observation in mind.

The single most important line in the cell below is the one that builds the `tool_result` block — that's the bridge from "real computer state" back into Claude's context. In a production setup the `content` of that block would be the actual screenshot bytes; here it's just text.


In [ ]:
# Reuse the same schema from §7
tools = [computer_tool]

# --- Step 1: Claude requests an action ---
messages = [{"role": "user", "content": (
    "Open the Sign In page and tell me what fields it has."
)}]

first = client.messages.create(
    model=model,
    max_tokens=500,
    tools=tools,
    messages=messages,
)

print("--- Step 1: Claude's request ---")
print("stop_reason:", first.stop_reason)
for block in first.content:
    print(block)

# --- Step 2: Append assistant turn + simulated tool_result, send round 2 ---
messages.append({"role": "assistant", "content": first.content})

tool_use = next(b for b in first.content if b.type == "tool_use")
fake_observation = (
    "Screenshot description: a Sign In page is visible. It has two text inputs labeled "
    "'Email' and 'Password', a 'Forgot password?' link, and a blue 'Sign In' button at "
    "coordinates (420, 280)."
)

messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_use.id,
        "content": fake_observation,
    }],
})

second = client.messages.create(
    model=model,
    max_tokens=500,
    tools=tools,
    messages=messages,
)

print("\n--- Step 2: Claude's response after seeing the observation ---")
print("stop_reason:", second.stop_reason)
for block in second.content:
    print(block)


> **🏫 During class:**
> 1. Run the cell. Walk through Step 1 → tool_use, Step 2 → either a `text` answer (because Claude saw enough in our observation) or another `tool_use` (it wants to act again).
> 2. Point at the `tool_result` block: *"This is the hinge. In production, `content` is screenshot bytes that Claude reads visually. Here, it's a string we made up. The protocol is identical."*
> 3. Variation: change `fake_observation` to *"The page is blank — a network error popup is on top."* Re-run. Watch Claude's next move shift from "answer" to "dismiss the popup." That responsiveness to observed state is the whole point of Computer Use.
> 4. Tie back to §1: *"This is the loop we promised at the beginning. One API call became an autonomous agent — by adding tools, a runtime, and the discipline to re-send history every turn."*


---
# 9. Recap + practice exercises

### Recap (run through these out loud)
- **Anthropic Apps** = real apps Anthropic ships on top of the same SDK we're using. Studying them maps every "magic" feature to an SDK primitive.
- **Claude Code** = system prompt + file/shell/web tools + a loop. The model itself is unchanged.
- **The three-turn workflow** (identify → plan → implement) gets dramatically better code than a one-shot prompt.
- **MCP servers** are a transport for tool schemas. Underneath, it's the same `tools=` parameter you'd hand-write.
- **Parallelism** = independent message lists run concurrently. Worktrees apply that same idea to file state.
- **Automated debugging** = `messages.create()` + structured output (pre-fill + stop sequence) wrapped in a cron job.
- **Computer Use** = a tool schema plus a runtime that executes the actions. Claude never touches a computer directly.
- **The full agent loop** is two API calls + a `tool_result` glue block, repeated until done. That's it.

### Exercises (do the first in class, assign the rest)
1. **Mini Claude Code:** wrap the §3 three-turn workflow in a function `claude_code_workflow(task, files)` that prints all three turns. Stress-test it by running two tasks back-to-back with fresh `msgs` lists.
2. **Mock MCP server:** add a second tool to §4 (e.g., `web_search(query: str)`) and write a prompt that requires both tools in sequence. Inspect the second `tool_use` to confirm Claude chains them.
3. **Parallel summary:** given five distinct news headlines, run five parallel `chat()` calls each producing a 1-sentence summary, then a 6th synthesis call that reads all five summaries. Time end-to-end.
4. **Debug agent v2:** extend §6 to handle multiple unique errors at once. Schema: `{"errors": [{"error": ..., "fix": ...}, ...]}`. Verify your prompt forces a list even with a single error.
5. **Faux Computer Use loop:** wrap §8's two calls in a `while` loop. Hand-code three fake observations the loop should respond to in sequence ("login button visible" → "logged in" → "dashboard loaded"). Stop when Claude returns no `tool_use` block.


In [ ]:
# Exercise 1 scaffold — finish this live in class together.
# Uncomment to run.
#
# def claude_code_workflow(task: str, files: str) -> dict:
#     msgs = []
#     add_user_message(msgs, f"Files:\n{files}\nTask: {task}\nWhich files matter? No code yet.")
#     identify = chat(msgs); add_assistant_message(msgs, identify)
#
#     add_user_message(msgs, "Now propose a 5-bullet plan. Still no code.")
#     plan = chat(msgs); add_assistant_message(msgs, plan)
#
#     add_user_message(msgs, "Now write a minimal pseudo-code patch.")
#     patch = chat(msgs); add_assistant_message(msgs, patch)
#
#     return {"identify": identify, "plan": plan, "patch": patch}
#
# result = claude_code_workflow(
#     task="Add audit logging to all writes",
#     files="- src/db.py\n- src/api.py\n- README.md",
# )
# for stage, content in result.items():
#     print(f"=== {stage.upper()} ===\n{content}\n")
